# Lesson 12｜Python 小專案：個人記帳工具

以一個可完成的專案整合 List、Dict、迴圈、函式、錯誤處理、Path 與 CSV。先完成最小版本，再逐步加入統計、搜尋與報表。


## 解答 01｜建立交易資料

先用 List 與 Dict 表示三筆收入／支出。


In [ ]:
records = [
    {'date': '2026-08-01', 'category': '餐飲', 'amount': -120, 'note': '午餐'},
    {'date': '2026-08-01', 'category': '薪資', 'amount': 30000, 'note': '薪水'},
    {'date': '2026-08-02', 'category': '交通', 'amount': -50, 'note': '捷運'},
]
print('筆數：', len(records))


## 解答 02｜逐筆顯示

使用 for 迴圈印出日期、分類與金額。


In [ ]:
for record in records:
    print(record['date'], record['category'], record['amount'])


## 解答 03｜計算總餘額

使用迴圈計算所有金額。


In [ ]:
balance = 0
for record in records:
    balance += record['amount']
print('目前餘額：', balance)


## 解答 04｜分開收入與支出

以條件判斷分別加總正數與負數。


In [ ]:
income = 0
expense = 0
for record in records:
    if record['amount'] >= 0:
        income += record['amount']
    else:
        expense += abs(record['amount'])
print('收入：', income)
print('支出：', expense)


## 解答 05｜新增交易函式

用函式集中建立資料的規則。


In [ ]:
def add_record(records, date, category, amount, note):
    record = {'date': date, 'category': category, 'amount': amount, 'note': note}
    records.append(record)
    return record

new_record = add_record(records, '2026-08-03', '購物', -350, '生活用品')
print('新增：', new_record)


## 解答 06｜驗證空白分類

分類不可為空白，否則主動 raise。


In [ ]:
def validate_category(category):
    if category.strip() == '':
        raise ValueError('分類不可空白')
    return category.strip()

print(validate_category(' 餐飲 '))


## 解答 07｜轉換金額

將文字金額安全轉成 float。


In [ ]:
def parse_amount(text):
    try:
        return float(text)
    except ValueError:
        print('金額格式錯誤')
        return None

print(parse_amount('250'))


## 解答 08｜顯示單筆交易

用函式建立一致的輸出格式。


In [ ]:
def format_record(record):
    return f"{record['date']}｜{record['category']}｜{record['amount']}｜{record['note']}"

print(format_record(records[0]))


## 解答 09｜顯示全部交易

重複使用 format_record。


In [ ]:
def show_records(records):
    if not records:
        print('目前沒有資料')
        return
    for index, record in enumerate(records, 1):
        print(index, format_record(record))

show_records(records)


## 解答 10｜分類支出統計

使用 Dict 累加各分類支出。


In [ ]:
def expense_by_category(records):
    summary = {}
    for record in records:
        if record['amount'] < 0:
            category = record['category']
            summary[category] = summary.get(category, 0) + abs(record['amount'])
    return summary

print(expense_by_category(records))


## 解答 11｜找出最大支出

只比較支出資料，輸出金額最高的一筆。


In [ ]:
largest = None
for record in records:
    if record['amount'] < 0:
        if largest is None or record['amount'] < largest['amount']:
            largest = record
if largest is not None:
    print('最大支出：', format_record(largest))
else:
    print('沒有支出資料')


## 解答 12｜依日期篩選

寫函式找出指定日期的交易。


In [ ]:
def filter_by_date(records, target_date):
    result = []
    for record in records:
        if record['date'] == target_date:
            result.append(record)
    return result

show_records(filter_by_date(records, '2026-08-01'))


## 解答 13｜依分類篩選

不分前後空白搜尋指定分類。


In [ ]:
def filter_by_category(records, target):
    target = target.strip()
    result = []
    for record in records:
        if record['category'] == target:
            result.append(record)
    return result

show_records(filter_by_category(records, '餐飲'))


## 解答 14｜準備 CSV 路徑

建立 data 資料夾與檔案路徑。


In [ ]:
from pathlib import Path
data_dir = Path('lesson12_project_data')
data_dir.mkdir(exist_ok=True)
csv_path = data_dir / 'records.csv'
print('資料位置：', csv_path.resolve())


## 解答 15｜寫入 CSV

使用 DictWriter 保存交易。


In [ ]:
import csv

with csv_path.open('w', newline='', encoding='utf-8-sig') as file:
    writer = csv.DictWriter(file, fieldnames=['date', 'category', 'amount', 'note'])
    writer.writeheader()
    writer.writerows(records)
print('已保存：', csv_path)


## 解答 16｜讀取 CSV

讀取後把 amount 轉回 float。


In [ ]:
def load_records(path):
    loaded = []
    if not path.exists():
        return loaded
    with path.open(encoding='utf-8-sig') as file:
        for row in csv.DictReader(file):
            row['amount'] = float(row['amount'])
            loaded.append(row)
    return loaded

loaded_records = load_records(csv_path)
show_records(loaded_records)


## 解答 17｜保存函式

把 CSV 寫入步驟整理成可重複使用的函式。


In [ ]:
def save_records(records, path):
    with path.open('w', newline='', encoding='utf-8-sig') as file:
        writer = csv.DictWriter(file, fieldnames=['date', 'category', 'amount', 'note'])
        writer.writeheader()
        writer.writerows(records)
    return len(records)

print('保存筆數：', save_records(records, csv_path))


## 解答 18｜摘要函式

回傳收入、支出與餘額。


In [ ]:
def calculate_summary(records):
    income = 0
    expense = 0
    for record in records:
        if record['amount'] >= 0:
            income += record['amount']
        else:
            expense += abs(record['amount'])
    return {'income': income, 'expense': expense, 'balance': income - expense}

summary = calculate_summary(records)
print('收入：', summary['income'])
print('支出：', summary['expense'])
print('餘額：', summary['balance'])


## 解答 19｜刪除交易

使用編號刪除前先驗證範圍。


In [ ]:
def delete_record(records, number):
    index = number - 1
    if index < 0 or index >= len(records):
        raise IndexError('交易編號不存在')
    return records.pop(index)

demo_records = records.copy()
removed = delete_record(demo_records, 1)
print('刪除：', format_record(removed))


## 解答 20｜選單畫面

建立清楚的文字選單。


In [ ]:
def show_menu():
    print('1. 新增交易')
    print('2. 查看交易')
    print('3. 查看摘要')
    print('4. 保存資料')
    print('0. 結束')

show_menu()


## 解答 21｜處理選單選擇

用 Dict 對照選項與功能名稱。


In [ ]:
menu = {'1': '新增交易', '2': '查看交易', '3': '查看摘要', '4': '保存資料', '0': '結束'}
choice = '3'
if choice in menu:
    print('你選擇：', menu[choice])
else:
    print('沒有這個選項')


## 解答 22｜輸入日期驗證

使用 datetime 檢查 YYYY-MM-DD。


In [ ]:
from datetime import datetime

def validate_date(text):
    try:
        datetime.strptime(text, '%Y-%m-%d')
        return True
    except ValueError:
        return False

print(validate_date('2026-08-15'))
print(validate_date('2026/08/15'))


## 解答 23｜產生文字報表

把摘要整理成容易閱讀的文字。


In [ ]:
def build_report(records):
    summary = calculate_summary(records)
    lines = [
        '記帳摘要',
        f"收入：{summary['income']:.0f}",
        f"支出：{summary['expense']:.0f}",
        f"餘額：{summary['balance']:.0f}",
    ]
    return '\n'.join(lines)

print(build_report(records))


## 解答 24｜輸出報表檔案

把摘要保存成 summary.txt。


In [ ]:
report_path = data_dir / 'summary.txt'
report_path.write_text(build_report(records), encoding='utf-8')
print('報表位置：', report_path.resolve())


## 解答 25｜整合專案流程

載入資料、加入一筆、保存並顯示摘要。


In [ ]:
project_records = load_records(csv_path)
add_record(project_records, '2026-08-15', '餐飲', -180, '晚餐')
save_records(project_records, csv_path)
show_records(project_records)
print(build_report(project_records))
